# AI-Powered Study Assistant — RAG Pipeline

## Problem Statement
The AI-Powered Study Assistant is an intelligent system that helps students interact with
their study materials efficiently. Using Retrieval-Augmented Generation (RAG), it allows
users to upload documents and ask questions in natural language. The system processes
documents by extracting, chunking, and converting text into embeddings stored in a vector
database. When a query is asked, it retrieves relevant content and generates accurate,
context-based answers using a language model.

## Objectives
- Enable question answering over custom documents (PDF, TXT)
- Retrieve only relevant, grounded context for each question
- Validate the system with multiple, varied test questions



### IMPORTS

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers import EnsembleRetriever, BM25Retriever
import time

print("Imports successful")

C:\Users\LOQ\AppData\Local\Temp\ipykernel_15636\2686919378.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


Imports successful


## Step 1: Document Ingestion
A flexible loader that supports both PDF and TXT files, based on file extension.

In [3]:
def load_document(file_path):
    """Loads a document based on its file extension. Supports .pdf and .txt for now."""
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {ext}. Supported types: .pdf, .txt")
    
    docs = loader.load()
    print(f"Loaded {len(docs)} document section(s) from {os.path.basename(file_path)}")
    return docs

In [4]:
def load_new_document(file_path):
    """Rebuilds the entire pipeline for a new document — chunks, embeddings, vector store, retriever."""
    global documents, chunks, vector_store, retriever, pdf_path
    
    pdf_path = file_path
    documents = load_document(file_path)
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    
    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    
    print(f"Ready. Loaded {len(documents)} section(s), {len(chunks)} chunks from {file_path}")

In [9]:
pdf_path = r"C:\Users\LOQ\Documents\GitHub\CSI_INTERNSHIP_2026\Final_Project\unit4 dl.pdf"
load_new_document(pdf_path)
print(documents[0].page_content[:500])

Loaded 61 document section(s) from unit4 dl.pdf
Ready. Loaded 61 section(s), 82 chunks from C:\Users\LOQ\Documents\GitHub\CSI_INTERNSHIP_2026\Final_Project\unit4 dl.pdf
CSN 342
Deep Learning
1
Faculty & Course coordinator : 
Dr. Anushikha Singh


## Step 2: Text Chunking
Split the extracted text into smaller, overlapping chunks so retrieval can work on focused, manageable pieces of context.

In [6]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
#chunk_size=500 — each chunk will be roughly 500 characters 
#chunk_overlap=50 — each chunk shares the last 50 characters with the next chunk
chunks= text_splitter.split_documents(documents)

## Step 3: Embedding Creation
Convert text into dense vector representations using a pretrained sentence embedding model, enabling semantic similarity search.

In [7]:
embeddings= HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")# convert text chunks into vector embeddings
test_vector=embeddings.embed_query("What is RAG?")
print(f"Vector length: {len(test_vector)}")
print(test_vector[:10])  # print first 10 dimensions of the vector

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector length: 384
[-0.06957073509693146, 0.09520001709461212, 0.01602139137685299, 0.006801468785852194, -0.08840496838092804, 0.014204825274646282, 0.0540277361869812, 0.045636896044015884, -0.03192176669836044, -0.029563721269369125]


## Step 4: Vector Database
Store all chunk embeddings in a FAISS vector index for fast similarity search at query time.

In [8]:
vector_store=FAISS.from_documents(chunks,embeddings)
print("Vector store created successfully")
print(f"Number of vectors in the store: {vector_store.index.ntotal}")

Vector store created successfully
Number of vectors in the store: 82


## Step 5: Query Embedding and Retrieval
Convert a test question into an embedding and retrieve the top-k most similar chunks from the vector store.

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

query = "What is CNN?"
relevant_chunks = retriever.invoke(query)

print(f"Retrieved {len(relevant_chunks)} chunks\n")
for i, chunk in enumerate(relevant_chunks):
    print(f"--- Chunk {i+1} ---")
    print(chunk.page_content)
    print()

Retrieved 3 chunks

--- Chunk 1 ---
Convolutional Neural Network
• Convolutional Neural Network (CNN) is the
extended version of artificial neural networks
(ANN) which is predominantly used to extract the
feature from the grid-like matrix dataset. For
example visual datasets like images or videos where
data patterns play an extensive role.
• ANN takes vectors as input so there is a need to
convert images in the vector. Spatial information is
lost in this process.
• CNN leading to sparse connections between input

--- Chunk 2 ---
How does a Convolutional Neural 
Network (CNN) work?
• A convolutional neural network, or ConvNet, is just a neural 
network that uses convolution. 
• Convolution is a mathematical operation that allows the 
merging of two sets of information. In the case of CNN, 
convolution is applied to the input data to filter the 
information and produce a feature map.
• This filter is also called a kernel, or feature detector, and its 
dimensions can be, for example, 3x3.

## Step 6: Answer Generation
Combine the retrieved chunks with the user's question into a single prompt, and generate a grounded answer using the local LLM. The model is instructed to answer only from the given context, and to say so explicitly if the answer isn't present.

In [15]:
llm = ChatOllama(model="llama3.1:8b",temperature=0)
def ask(question,verbose=True):
    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)     
    return response.content

Test the full pipeline end-to-end with a single question.

In [16]:
ask("What is CNN?") #This part can say that the answer is not in the document if it is not found in the context. Because I am asking the fixed question for every document , this is only fo rthe testing that is is really working or not.

"A Convolutional Neural Network (CNN) is an extended version of artificial neural networks (ANN). It is predominantly used to extract features from grid-like matrix datasets, such as visual data like images or videos where data patterns play a significant role. In other words, it's a type of neural network that specializes in processing and analyzing visual data by extracting relevant features from the input data."

## Step 7: Validation with Multiple Sample Questions
To confirm the pipeline generalizes beyond a single query, the system is tested with several different questions 

In [17]:
import time

test_questions = [
    "What is Stride?",
    "What is Pooling?",
    "Lyers used to build ConvNets",
    "What is the meaning of King?",
]

validation_log = []

for q in test_questions:
    start = time.time()
    answer = ask(q, verbose=False)
    elapsed = time.time() - start
    validation_log.append({"question": q, "answer": answer, "time_sec": round(elapsed, 2)})
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Response time: {elapsed:.2f}s")
    print("-" * 80)
    #We can See the response time for each question and the answer provided by the model. This will help us validate if the model is working correctly and efficiently with the given document context.
    # can give the ans as "I don't know based on the provided document." if the answer is not found in the context. (Because i am asking the fixed question for every document , this is only for the testing that is it really working or not.)

Q: What is Stride?
A: Stride refers to the number of pixels that we slide over the input image by the kernel. In other words, it's the distance or step size at which the kernel moves across the image when performing operations such as convolution or pooling. By adjusting the stride, we can control how much of the image is processed and how quickly the computation proceeds. A larger stride means fewer pixels are considered in each operation, resulting in a smaller output volume that requires less memory and computation time, but may also lead to loss of detail.
Response time: 3.10s
--------------------------------------------------------------------------------
Q: What is Pooling?
A: Pooling refers to a component of a Convolutional Neural Network (CNN) that reduces the spatial dimensions of feature maps, thereby decreasing the amount of data and parameters. It's also known as a downsampling layer. The main function of pooling is to reduce the size of the volume, making computation faste

## Step 8: Optimization Experiment 1 — Chunking Strategy
Three chunking configurations (baseline, smaller with more overlap, larger) are each used to build a separate vector store and retriever, so retrieval quality can be compared directly rather than just chunk statistics.

In [18]:
chunk_configs = [
    {"chunk_size": 500, "chunk_overlap": 50, "label": "baseline"},
    {"chunk_size": 300, "chunk_overlap": 75, "label": "smaller"},
    {"chunk_size": 800, "chunk_overlap": 100, "label": "larger"},
]

chunking_stores = {}
chunking_retrievers = {}

for cfg in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(chunk_size=cfg["chunk_size"], chunk_overlap=cfg["chunk_overlap"])
    exp_chunks = splitter.split_documents(documents)

    exp_store = FAISS.from_documents(exp_chunks, embeddings)
    exp_retriever = exp_store.as_retriever(search_kwargs={"k": 3})

    chunking_stores[cfg["label"]] = {
        "chunk_size": cfg["chunk_size"],
        "chunk_overlap": cfg["chunk_overlap"],
        "num_chunks": len(exp_chunks),
        "avg_chunk_len": round(sum(len(c.page_content) for c in exp_chunks) / len(exp_chunks), 1)
    }
    chunking_retrievers[cfg["label"]] = exp_retriever

for label, stats in chunking_stores.items():
    print(label, stats)

baseline {'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82, 'avg_chunk_len': 246.9}
smaller {'chunk_size': 300, 'chunk_overlap': 75, 'num_chunks': 118, 'avg_chunk_len': 195.3}
larger {'chunk_size': 800, 'chunk_overlap': 100, 'num_chunks': 67, 'avg_chunk_len': 303.9}


A helper function to ask a question using a specific chunking configuration's retriever.

In [19]:
def ask_with_config(question, label, verbose=True):
    retriever_cfg = chunking_retrievers[label]
    relevant_chunks = retriever_cfg.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    if verbose:
        print(response.content)
    return response.content

Run the same test query through all three chunking configurations and compare the real generated answers.

In [20]:
test_query = "what are the Benefits of Transfer Learning ?"

for label in chunking_retrievers:
    print(f"--- Config: {label} (size={chunking_stores[label]['chunk_size']}, overlap={chunking_stores[label]['chunk_overlap']}) ---")
    answer = ask_with_config(test_query, label, verbose=False)
    print(answer)
    print("-" * 80)
    # The same is repeated for each chunking configuration, allowing us to compare how different chunk sizes and overlaps affect the retrieval and answer quality.

--- Config: baseline (size=500, overlap=50) ---
The benefits of Transfer Learning include:

Two key challenges that Transfer Learning offers solutions to are:

1. **Limited Data**: Acquiring extensive labelled data is often challenging and costly, but Transfer Learning enables us to use pre-trained models, reducing our dependency on large datasets.

2. **Enhanced Performance**: Starting with a pre-trained model which has already learned from substantial data allows for faster and more accurate results on new tasks, ideal for applications needing high accuracy and efficiency.
--------------------------------------------------------------------------------
--- Config: smaller (size=300, overlap=75) ---
The benefits of transfer learning include:

1. **Reducing the risk of overfitting**: By incorporating generalizable features from the initial task, the model is less likely to overfit on the new task, which means it will perform better on unseen data.

2. **Accelerating learning and improv

## Step 9: Optimization Experiment 2 — Hybrid Search
Pure vector search can miss chunks that share exact keywords with the query but aren't the closest semantic match. A hybrid retriever combining BM25 (keyword-based) and FAISS (vector-based) is built here and compared against vector-only retrieval.

In [21]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6]
)

query = "Layers Used to Build ConvNet?"

vector_only_results = vector_retriever.invoke(query)
hybrid_results = hybrid_retriever.invoke(query)

print(f"Vector-only retrieved: {len(vector_only_results)} chunks")
print(f"Hybrid retrieved: {len(hybrid_results)} chunks")

Vector-only retrieved: 3 chunks
Hybrid retrieved: 5 chunks


A helper function to ask a question using the hybrid retriever.

In [22]:
def ask_hybrid(question, verbose=True):
    relevant_chunks = hybrid_retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])

    prompt = f"""You are answering questions using only the context below.
Give a detailed, well-explained answer — don't just list terms, explain each one briefly using the information given in the context.
Do not mention or comment on topics that were not asked about, even if they appear in the context.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:"""

    response = llm.invoke(prompt)
    if verbose:
        print(response.content)
    return response.content

## Step 10: Conversation Memory
A sliding-window memory (last 3 turns) is added so follow-up questions can reference earlier parts of the conversation, instead of every question being treated in isolation.

In [23]:
chat_history = []
MAX_HISTORY_TURNS = 3

def ask_hybrid_with_memory(question, verbose=False):
    relevant_chunks = hybrid_retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    recent_history = chat_history[-MAX_HISTORY_TURNS:]
    history_text = "\n\n".join([f"Q: {q}\nA: {a}" for q, a in recent_history])
    
    prompt = f"""You are answering questions using only the context below.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."
    
Previous conversation:
{history_text}

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)
    chat_history.append((question, response.content))
    
    if verbose:
        print(f"Q: {question}")
        print(f"A: {response.content}")
    return response.content

In [24]:
chat_history = []
MAX_HISTORY_TURNS = 3

def ask_with_memory(question, verbose=False):
    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([chunk.page_content for chunk in relevant_chunks])
    
    recent_history = chat_history[-MAX_HISTORY_TURNS:]
    history_text = "\n\n".join([f"Q: {q}\nA: {a}" for q, a in recent_history])
    
    prompt = f"""You are answering questions using only the context below.
If the context genuinely does not contain the answer, say "I don't know based on the provided document."

Previous conversation:
{history_text}

Context:
{context}

Question: {question}

Answer:"""
    
    response = llm.invoke(prompt)
    chat_history.append((question, response.content))
    
    if verbose:
        print(response.content)
    return response.content

Test memory with a follow-up question that depends on the previous answer.

In [26]:
ask_with_memory("what is Transfer learning?")
print("\n" + "="*80 + "\n")
ask_with_memory("how It is usefull?")

'Transfer learning is useful because it shortens training time and conserves resources by utilizing existing models, hence eliminating the need for training from scratch. Additionally, it allows models trained on one task to be fine-tuned for related tasks, making transfer learning versatile for various applications from image recognition to natural language processing.'

## Step 11: Retrieval Strategy Comparison
Run the same question through vector-only retrieval and hybrid retrieval side by side, to compare their generated answers directly.

In [27]:
print("--- Vector-only ---")
vector_answer = ask("Layers Used to Build ConvNet?", verbose=False)
print(vector_answer)
print("-" * 80)

print("--- Hybrid (BM25 + FAISS) ---")
hybrid_answer = ask_hybrid("Layers Used to Build ConvNet?", verbose=False)
print(hybrid_answer)

--- Vector-only ---
A complete Convolutional Neural Network (ConvNet) architecture, also known as a covnet, is built using a sequence of layers. The types of layers used to build a ConvNet include:

1. **Input Layer**: This layer receives the raw input data, which in this case is an image with dimensions 32 x 32 x 3. It holds the input image and prepares it for processing by the subsequent layers.

2. **Convolutional Layers**: These layers are used to extract features from the input dataset. They apply learnable filters (kernels) to the input images, allowing the network to detect patterns and features in the data.

3. **Activation Layer**: This layer adds nonlinearity to the network by applying an element-wise activation function to the output of the convolutional layer. Common activation functions include RELU, Tanh, and Leaky RELU. The activation layer does not change the dimensions of the volume; it remains 32 x 32 x 12.

4. **Pooling Layer**: This layer is periodically inserted in

## Step 12: System Metrics Report
A consolidated summary of the system's configuration: document source, chunking profiles tested, embedding model and dimensions, vector store details, retrieval strategies compared, and the LLM setup used for generation.

In [29]:
metrics_report = {
    "document_source": os.path.basename(pdf_path),
    "num_pages_loaded": len(documents),
    "chunking": {
        "chunk_size": 500,
        "chunk_overlap": 50,
        "num_chunks": len(chunks)
    },
    "chunking_configs_tested": [
        {
            "label": label,
            "chunk_size": stats["chunk_size"],
            "chunk_overlap": stats["chunk_overlap"],
            "num_chunks": stats["num_chunks"]
        }
        for label, stats in chunking_stores.items()
    ],
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dimensions": len(test_vector),
    "vector_store": {
        "type": "FAISS",
        "num_vectors": vector_store.index.ntotal
    },
    "retrieval": {
        "top_k": 3,
        "strategies_tested": [
            "vector-only (FAISS)",
            "hybrid (BM25 + FAISS, weights 0.4/0.6)"
        ]
    },
    "llm": {
        "provider": "Ollama",
        "model": "llama3.1:8b",
        "temperature": 0
    },
    "validation_questions_tested": len(test_questions)
}

for k, v in metrics_report.items():
    print(f"{k}: {v}")

document_source: unit4 dl.pdf
num_pages_loaded: 61
chunking: {'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82}
chunking_configs_tested: [{'label': 'baseline', 'chunk_size': 500, 'chunk_overlap': 50, 'num_chunks': 82}, {'label': 'smaller', 'chunk_size': 300, 'chunk_overlap': 75, 'num_chunks': 118}, {'label': 'larger', 'chunk_size': 800, 'chunk_overlap': 100, 'num_chunks': 67}]
embedding_model: sentence-transformers/all-MiniLM-L6-v2
embedding_dimensions: 384
vector_store: {'type': 'FAISS', 'num_vectors': 82}
retrieval: {'top_k': 3, 'strategies_tested': ['vector-only (FAISS)', 'hybrid (BM25 + FAISS, weights 0.4/0.6)']}
llm: {'provider': 'Ollama', 'model': 'llama3.1:8b', 'temperature': 0}
validation_questions_tested: 4


## Step 13: Interactive Chat Loop
A simple live chat interface for freely asking the system anything about the loaded
document, using conversation memory. Type `exit`, `quit`, or `bye` to end.


In [30]:
print("=" * 60)
print("RAG Chatbot")
print("Type 'exit, bye, quit' to end the conversation.")
print("=" * 60)

while True:
    question = input("\nYou: ")
    print("You:", question)

    if question.lower() in ["exit", "quit", "bye"]:
        print("Bot: Goodbye!")
        break

    answer = ask_hybrid_with_memory(question, verbose=False)
    print("\nBot:", answer)

RAG Chatbot
Type 'exit, bye, quit' to end the conversation.
You: What is the meaning of transfer learning?

Bot: Transfer learning is a technique in deep learning where pre-trained models trained on large-scale datasets are used to solve new tasks with limited labeled data. It's also a machine learning technique where a model trained on one task is repurposed as the foundation for a second task, beneficial when the second task is related to the first or when data for the second task is limited.
You: what are the benefits of it?

Bot: The benefits of Transfer Learning mentioned in the context are:

* It shortens training time.
* It conserves resources by utilizing existing models, hence eliminating the need for training from scratch.
* It allows models trained on one task to be fine-tuned for related tasks.

Additionally, it is stated that fine-tuning a network with transfer learning is usually much faster and easier than training a network with randomly initialized weights from scratch

## PROBLEM ##

I have tried to make the project interactive using Streamlit, but I am currently facing an issue where the application only displays a black screen. I am working on resolving this issue so that I can demonstrate the fully interactive application during the final evaluation call.

For now, I will create a separate file containing screenshots that demonstrate how the Project works and how the interactive features are implemented

## Conclusion

This project implements a complete RAG pipeline — document ingestion (PDF/TXT), chunking,
embedding, vector storage, retrieval, and grounded answer generation — with two optimization
experiments (chunking comparison and hybrid search) tested against real queries rather than
statistics alone. The system was validated across multiple questions, including an
out-of-scope question to confirm faithful, non-hallucinated behavior, and supports multi-turn
conversation through a sliding-window memory.

## Limitations
- No persistent memory across separate notebook sessions (resets when the kernel restarts)
- Runs on a local 8B-parameter model, which has less reasoning depth than larger hosted models
- Single-document scope; no cross-document retrieval

## Future Work
- Extend into a Streamlit interface with file upload and chat UI
- Add a re-ranking layer on top of hybrid retrieval
- Explore summarization and voice interaction as additional features
